# VORQ · 01 — Knowledge Base & Entity Resolver

This notebook contains the VORQ seed knowledge base and entity resolver.
Run all cells to initialise the KB and verify entity resolution.

## 1. Seed Knowledge Base (JSON)
Loaded from `vorq/data/kb/seed_knowledge_base.json` at runtime.

In [ ]:
import json, os, re
from typing import Optional, Dict, List
from pathlib import Path

# Resolve paths relative to this notebook
REPO_ROOT = Path(os.getcwd()).parent
KB_DIR    = REPO_ROOT / 'vorq' / 'data' / 'kb'
SEED_PATH = KB_DIR / 'seed_knowledge_base.json'

print(f'KB path: {SEED_PATH}')
print(f'Exists: {SEED_PATH.exists()}')

In [ ]:
with open(SEED_PATH) as f:
    KB = json.load(f)

print('Countries:  ', len(KB['countries']))
print('Industries: ', len(KB['industries']))
print('Companies:  ', len(KB['companies']))
print('Commodities:', len(KB['commodities']))

## 2. Entity Resolver

In [ ]:
class EntityResolver:
    """Fast, deterministic entity resolution against the VORQ seed KB."""

    def __init__(self, kb_path: str = str(SEED_PATH)):
        with open(kb_path) as f:
            self._kb = json.load(f)
        self._country_index:  Dict[str, dict] = {}
        self._industry_index: Dict[str, dict] = {}
        self._company_index:  Dict[str, dict] = {}

        # ── country aliases ─────────────────────────────────────
        _ca = {
            'united states':'US','usa':'US','us':'US','america':'US',
            'china':'CN','prc':'CN','peoples republic':'CN',
            'india':'IN','bharat':'IN','germany':'DE','japan':'JP','nippon':'JP',
            'taiwan':'TW','roc':'TW','south korea':'KR','korea':'KR',
            'saudi arabia':'SA','ksa':'SA','australia':'AU','russia':'RU',
            'brazil':'BR','united kingdom':'GB','uk':'GB','britain':'GB',
            'france':'FR','netherlands':'NL','holland':'NL','denmark':'DK',
            'chile':'CL','peru':'PE','qatar':'QA','mexico':'MX','canada':'CA',
            'poland':'PL','uae':'AE','united arab emirates':'AE',
        }
        self._country_aliases = {k.lower(): v for k, v in _ca.items()}
        for c in self._kb['countries']:
            self._country_index[c['id']] = c
            self._country_aliases[c['name'].lower()] = c['id']

        # ── industry aliases ─────────────────────────────────────
        _ia = {
            'tech':'technology','it':'technology','semiconductor':'technology',
            'semiconductors':'technology','chips':'technology','electronics':'technology',
            'pharma':'healthcare','pharmaceutical':'healthcare','pharmaceuticals':'healthcare',
            'medical':'healthcare','drugs':'healthcare','oil':'energy','gas':'energy',
            'petroleum':'energy','oil and gas':'energy','crude':'energy','lng':'energy',
            'military':'defense','aerospace':'defense','weapons':'defense',
            'auto':'automotive','cars':'automotive','vehicles':'automotive',
            'factory':'manufacturing','factories':'manufacturing','production':'manufacturing',
            'food':'agriculture','farming':'agriculture','crops':'agriculture',
            'metals':'mining','ore':'mining','minerals':'mining',
            'logistics':'shipping','freight':'shipping','transport':'shipping',
            'banking':'finance','banks':'finance','financial':'finance',
            'insurance':'finance','investment':'finance',
        }
        self._industry_aliases = {k.lower(): v for k, v in _ia.items()}
        for ind in self._kb['industries']:
            self._industry_index[ind['id']] = ind
            self._industry_aliases[ind['label'].lower()] = ind['id']

        # ── company aliases ──────────────────────────────────────
        _coa = {
            'apple':'AAPL','microsoft':'MSFT','google':'GOOGL','alphabet':'GOOGL',
            'nvidia':'NVDA','tsmc':'TSMC','taiwan semi':'TSMC',
            'samsung':'SAMSUNG_ELEC','asml':'ASML','jpmorgan':'JPM','jp morgan':'JPM',
            'aramco':'ARAMCO','saudi aramco':'ARAMCO','exxon':'XOM','exxonmobil':'XOM',
            'pfizer':'PFE','sun pharma':'SUN_PHARMA','toyota':'TOYOTA',
            'volkswagen':'VW','vw':'VW','basf':'BASF','nippon steel':'NIPPON_STEEL',
            'foxconn':'FOXCONN','intel':'INTC','maersk':'MAERSK','catl':'CATL',
        }
        self._company_aliases = {k.lower(): v for k, v in _coa.items()}
        for comp in self._kb['companies']:
            self._company_index[comp['id']] = comp
            self._company_aliases[comp['name'].lower()] = comp['id']

    # ── Public API ───────────────────────────────────────────────
    def resolve_country(self, text: str) -> Optional[dict]:
        key = text.strip().lower()
        cid = self._country_aliases.get(key)
        if cid: return self._country_index.get(cid)
        for alias, cid in self._country_aliases.items():
            if alias in key or key in alias: return self._country_index.get(cid)
        return None

    def resolve_industry(self, text: str) -> Optional[dict]:
        key = text.strip().lower()
        iid = self._industry_aliases.get(key)
        if iid: return self._industry_index.get(iid)
        for alias, iid in self._industry_aliases.items():
            if alias in key or key in alias: return self._industry_index.get(iid)
        return None

    def resolve_all(self, text: str) -> Dict[str, List[dict]]:
        words   = re.split(r'[\s,;.!?]+', text.lower())
        bigrams = [f'{words[i]} {words[i+1]}' for i in range(len(words)-1)]
        tokens  = words + bigrams
        countries, industries, companies = [], [], []
        seen_c, seen_i, seen_co = set(), set(), set()
        for token in tokens:
            cid = self._country_aliases.get(token)
            if cid and cid not in seen_c:
                rec = self._country_index.get(cid)
                if rec: countries.append(rec); seen_c.add(cid)
            iid = self._industry_aliases.get(token)
            if iid and iid not in seen_i:
                rec = self._industry_index.get(iid)
                if rec: industries.append(rec); seen_i.add(iid)
            comp_id = self._company_aliases.get(token)
            if comp_id and comp_id not in seen_co:
                rec = self._company_index.get(comp_id)
                if rec: companies.append(rec); seen_co.add(comp_id)
        return {'countries': countries, 'industries': industries, 'companies': companies}

    def get_companies_by_country(self, country_id: str) -> List[dict]:
        return [c for c in self._kb['companies'] if c['country'] == country_id]

    def get_companies_by_industry(self, industry_id: str) -> List[dict]:
        return [c for c in self._kb['companies'] if c['industry'] == industry_id]

    @property
    def kb(self): return self._kb


print('EntityResolver defined ✓')

## 3. Smoke Test

In [ ]:
resolver = EntityResolver()

# Test scenarios
tests = [
    'War between India and China in 1 year',
    'US sanctions on semiconductor exports to China',
    'Major oil supply shock in the Middle East',
]

for t in tests:
    ents = resolver.resolve_all(t)
    print(f'\n"{t}"')
    print(f'  Countries:  {[c["name"] for c in ents["countries"]]}')
    print(f'  Industries: {[i["label"] for i in ents["industries"]]}')
    print(f'  Companies:  {[c["name"] for c in ents["companies"]]}')